# Universe Filter Playground
Tier 1 weekly filter: large-cap US stocks → `data/watchlist.csv`

In [1]:
# Cell 1 — Setup & import
import sys
sys.path.insert(0, 'backend/01_scanner')

from universe_filter import (
    get_max_share_price,
    get_earnings_tickers,
    run_universe_filter,
    save_watchlist,
)
import pandas as pd

print('Imports OK')

Imports OK


In [2]:
# Cell 2 — Happy path: run full filter and inspect output
count = run_universe_filter()
print(f'\nResult: {count} stocks saved')

if count:
    df = pd.read_csv('data/watchlist.csv')
    display(df.head(10))
    print(f'\nColumns: {df.columns.tolist()}')
    print(f'Price range: ${df["price"].min():.2f} – ${df["price"].max():.2f}')
    print(f'ATR% range:  {df["atr_pct"].min():.1f}% – {df["atr_pct"].max():.1f}%')

[universe] Alpaca unavailable, using fallback ceiling $360.00: ('Key ID must be given to access Alpaca trade API', ' (env: APCA_API_KEY_ID)')
[universe] price ceiling: $360.00
[universe] querying TradingView screener...
[universe] 71 stocks after ATR% filter (1.0%–5.0%)
[universe] 1 stocks removed for upcoming earnings
[universe] saved 70 tickers to data/watchlist.csv

Result: 70 stocks saved


,ticker,price,volume,atr,atr_pct,rsi,sma20,sma50
0,NVDA,224.01,152105522,7.56,3.37,62.2,213.43,195.51
1,AAPL,302.19,42611433,5.93,1.96,74.4,285.88,268.61
2,PLTR,136.50,37861908,5.74,4.21,47.0,137.91,143.25
3,T,24.90,36317352,0.61,2.43,41.9,25.41,26.66
4,PFE,25.78,35461235,0.53,2.04,41.9,26.16,26.85
5,AMZN,264.17,35181153,6.81,2.58,57.0,266.27,239.65
6,BAC,51.18,34688420,1.14,2.22,48.3,51.79,50.68
7,CSCO,114.97,32812261,3.51,3.06,78.3,98.79,88.25
8,NFLX,88.22,32726935,2.60,2.95,42.0,89.57,93.91
9,WFC,75.65,19286154,1.93,2.55,42.6,77.74,79.01



Columns: ['ticker', 'price', 'volume', 'atr', 'atr_pct', 'rsi', 'sma20', 'sma50']
Price range: $24.90 – $332.62
ATR% range:  1.5% – 5.0%


In [3]:
# Cell 3 — Parameter variations

# Vary earnings window: how many stocks get removed at different lookforward windows?
for days in [3, 5, 10]:
    tickers = get_earnings_tickers(days_ahead=days)
    count = len(tickers) if tickers else 0
    print(f'days_ahead={days}: {count} earnings events')

# Show current price ceiling
print(f'\nCurrent price ceiling: ${get_max_share_price():.2f}')

days_ahead=3: 47 earnings events
days_ahead=5: 116 earnings events
days_ahead=10: 197 earnings events
[universe] Alpaca unavailable, using fallback ceiling $360.00: ('Key ID must be given to access Alpaca trade API', ' (env: APCA_API_KEY_ID)')

Current price ceiling: $360.00


In [4]:
# Cell 4 — Failure path: confirm graceful degradation
import os

# Temporarily remove Finnhub key — should skip earnings filter, not crash
original_key = os.environ.pop('FINNHUB_API_KEY', None)
result = get_earnings_tickers()
print(f'Missing Finnhub key → returned: {result}  (expected: None)')

# Restore key
if original_key:
    os.environ['FINNHUB_API_KEY'] = original_key

# Bad URL in earnings (simulate network failure)
import unittest.mock as mock
with mock.patch('requests.get', side_effect=Exception('network error')):
    result = get_earnings_tickers()
    print(f'Network failure → returned: {result}  (expected: None)')

[universe] FINNHUB_API_KEY not set, skipping earnings filter
Missing Finnhub key → returned: None  (expected: None)
[universe] Finnhub unavailable, skipping earnings filter: network error
Network failure → returned: None  (expected: None)


In [ ]:
# Cell 5 — Free play
